In [1]:
import sys
from pathlib import Path

PIPELINE_ROOT = Path(r"C:\streaming_emulator")

if str(PIPELINE_ROOT) not in sys.path:
    sys.path.insert(0, str(PIPELINE_ROOT))

print("sys.path[0]:", sys.path[0])


sys.path[0]: C:\streaming_emulator


## config_loader_py_test

In [2]:
from pathlib import Path
from ingest.app.config_loader import IngestConfig

cfg = IngestConfig(
    Path(r"C:\streaming_emulator\ingest\config\ingest_config.json")
)

print("Kafka:", cfg.kafka_bootstrap_servers)
print("Topics:", cfg.topic_mapping)
print("DLQ path:", cfg.dlq_path)
print("Metrics port:", cfg.metrics_port)


Kafka: localhost:9092
Topics: {'engine': 'telemetry.engine', 'transmission': 'telemetry.transmission', 'battery': 'telemetry.battery', 'tyre': 'telemetry.tyre', 'body': 'telemetry.body'}
DLQ path: ingest\dlq
Metrics port: 9101


## schemas_py_test

In [3]:
from ingest.app.schemas import IngestRequest

payload = {
    "metadata": {
        "row_hash": "abc123",
        "vehicle_id": "sim001",
        "module": "engine",
        "source_file": "engine.csv",
        "ingest_ts": "2024-07-05T08:00:00Z",
    },
    "data": {
        "engine_rpm_rpm": 1200,
        "engine_oil_temperature": 85.2,
    },
}

req = IngestRequest(**payload)
print("Validated OK:", type(req))


Validated OK: <class 'ingest.app.schemas.IngestRequest'>


In [4]:
bad_payload = {
    "metadata": {
        "vehicle_id": "sim001",
        "module": "engine",
        "source_file": "engine.csv",
    },
    "data": {},
}

IngestRequest(**bad_payload)


ValidationError: 1 validation error for IngestRequest
metadata.row_hash
  Field required [type=missing, input_value={'vehicle_id': 'sim001', ...rce_file': 'engine.csv'}, input_type=dict]
    For further information visit https://errors.pydantic.dev/2.12/v/missing

In [5]:
bad_payload = {
    "metadata": {
        "row_hash": "abc",
        "vehicle_id": "sim001",
        "module": "engine",
        "source_file": "engine.csv",
        "unexpected": "boom",
    },
    "data": {},
}

IngestRequest(**bad_payload)


ValidationError: 1 validation error for IngestRequest
metadata.unexpected
  Extra inputs are not permitted [type=extra_forbidden, input_value='boom', input_type=str]
    For further information visit https://errors.pydantic.dev/2.12/v/extra_forbidden

In [6]:
bad_payload = {
    "metadata": {
        "row_hash": "abc",
        "vehicle_id": "sim001",
        "module": "engine",
        "source_file": "engine.csv",
    },
    "data": ["not", "a", "dict"],
}

IngestRequest(**bad_payload)


ValidationError: 1 validation error for IngestRequest
data
  Input should be a valid dictionary [type=dict_type, input_value=['not', 'a', 'dict'], input_type=list]
    For further information visit https://errors.pydantic.dev/2.12/v/dict_type

## validator_py_test

In [7]:
from pathlib import Path
from replay.service.schema_loader import MasterSchema
from ingest.app.validator import DefensiveValidator

schema = MasterSchema(
    Path(r"C:\streaming_emulator\contracts\master.json")
)

validator = DefensiveValidator(schema)

valid_data = {
    "engine_rpm_rpm": 1200,
    "engine_oil_temperature": 85.2,
    # include ALL required engine columns in correct order
}

validated = validator.validate(
    module="engine",
    data=valid_data,
)

print("Validated OK:", type(validated))


IngestValidationError: Column order mismatch

In [8]:
validator.validate(
    module="wings",
    data={},
)


IngestValidationError: Unknown module

## idempotency_py_test

In [9]:
from ingest.app.idempotency import IdempotencyCache

cache = IdempotencyCache(ttl_seconds=60)

h = "abc123"

assert cache.seen_before(h) is False
cache.mark_seen(h)
assert cache.seen_before(h) is True


In [10]:
import time

cache = IdempotencyCache(ttl_seconds=1)

cache.mark_seen("x")
time.sleep(1.2)

assert cache.seen_before("x") is False


In [11]:
cache = IdempotencyCache(max_entries=2, ttl_seconds=60)

cache.mark_seen("a")
cache.mark_seen("b")
cache.mark_seen("c")

assert cache.seen_before("a") is False
assert cache.seen_before("b") is True
assert cache.seen_before("c") is True


## producer_py_test

In [12]:
import aiokafka
print(aiokafka.__version__)



0.13.0


In [13]:
from aiokafka import AIOKafkaProducer
from ingest.app.producer import KafkaProducerWrapper

producer = AIOKafkaProducer(
    bootstrap_servers="localhost:9092"
)
await producer.start()

wrapper = KafkaProducerWrapper(
    producer=producer,
    topic_map={
        "engine": "telemetry.engine"
    },
)

event = {
    "metadata": {
        "row_hash": "abc",
        "vehicle_id": "sim001",
        "module": "engine",
        "source_file": "x.csv",
        "ingest_ts": "2026-01-01T00:00:00Z",
    },
    "data": {
        "dummy": 1
    },
}

await wrapper.send(event=event)

await producer.stop()

print("Kafka send OK")


Kafka send OK


## dlq_py_test

In [17]:
from pathlib import Path
print("CWD:", Path.cwd())

CWD: C:\streaming_emulator\extras


In [20]:
from pathlib import Path
from ingest.app.dlq import IngestDLQWriter

PIPELINE_ROOT = Path(r"C:\streaming_emulator")

dlq = IngestDLQWriter(
    PIPELINE_ROOT / "ingest" / "dlq"
)

event = {
    "metadata": {
        "row_hash": "abc",
        "vehicle_id": "sim001",
        "module": "engine",
    },
    "data": {"x": 1},
}

dlq.write(
    error_type="schema_validation",
    error_message="Missing column",
    event=event,
    error_details={"column": "rpm"},
)


## metrics_py_test

In [21]:
from ingest.app.metrics import (
    ingest_requests_total,
    ingest_rows_accepted_total,
    export_metrics,
)

ingest_requests_total.inc()
ingest_rows_accepted_total.inc(2)

print(export_metrics().decode())


# HELP ingest_requests_total Total number of ingest HTTP requests received
# TYPE ingest_requests_total counter
ingest_requests_total 1.0
# HELP ingest_requests_created Total number of ingest HTTP requests received
# TYPE ingest_requests_created gauge
ingest_requests_created 1.767876538432728e+09
# HELP ingest_rows_accepted_total Total number of rows accepted for Kafka production
# TYPE ingest_rows_accepted_total counter
ingest_rows_accepted_total 2.0
# HELP ingest_rows_accepted_created Total number of rows accepted for Kafka production
# TYPE ingest_rows_accepted_created gauge
ingest_rows_accepted_created 1.7678765384327407e+09
# HELP ingest_rows_rejected_total Total number of rows rejected by ingest service
# TYPE ingest_rows_rejected_total counter
ingest_rows_rejected_total 0.0
# HELP ingest_rows_rejected_created Total number of rows rejected by ingest service
# TYPE ingest_rows_rejected_created gauge
ingest_rows_rejected_created 1.7678765384327476e+09
# HELP ingest_validation_laten

In [22]:
import time
from ingest.app.metrics import ingest_validation_latency_ms

with ingest_validation_latency_ms.time():
    time.sleep(0.02)

print("Recorded validation latency")


Recorded validation latency
